In [531]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMRegressor, LGBMClassifier
import sys
import re
from sklearn.preprocessing import LabelEncoder
import plotly.express as px
from scipy.stats import spearmanr

In [532]:
df_sold = pd.read_csv("../data/Sold/sold_transactions.csv")
df_sold.head()

C:\Users\mayab\AppData\Local\Temp\ipykernel_27752\4086709524.py:1: DtypeWarning: Columns (0: BuyerAgentAOR, 1: ListAgentAOR, 2: WaterfrontYN, 3: ListAgentEmail, 4: OriginatingSystemName, 5: OriginatingSystemSubName, 6: BuyerAgencyCompensationType, 7: latfilled, 8: lonfilled) have mixed types. Specify dtype option on import or set low_memory=False.
  df_sold = pd.read_csv("../data/Sold/sold_transactions.csv")


,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,WaterfrontYN,BasementYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,...,MiddleOrJuniorSchoolDistrict,OriginatingSystemName,OriginatingSystemSubName,BuyerAgencyCompensationType,BuyerAgencyCompensation,latfilled,lonfilled,Year,year_month,rate_30yr_fixed
0,Mlslistings,Mlslistings,"Carpet,Tile,Wood",True,NaN,NaN,False,499000.0,551985747,jwachter@cbnorcal.com,...,NaN,CRMLS,CRMLS_MLSL,NaN,NaN,NaN,NaN,2024,2024-01,6.6425
1,HighDesert,HighDesert,NaN,NaN,NaN,NaN,NaN,0.0,535486633,eabrown@lee-associates.com,...,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024,2024-01,6.6425
2,OrangeCounty,OrangeCounty,NaN,True,NaN,NaN,NaN,75000.0,529986282,Joe@9WINWIN.com,...,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024,2024-01,6.6425
3,InlandValleys,InlandValleys,NaN,True,NaN,NaN,NaN,199000.0,529618166,carolthefinder@yahoo.com,...,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024,2024-01,6.6425
4,SouthwestRiversideCounty,SouthwestRiversideCounty,NaN,True,NaN,NaN,NaN,19500.0,522614340,jtavisola@tavisola.com,...,NaN,CRMLS,CRMLS_CRM,NaN,NaN,NaN,NaN,2024,2024-01,6.6425


##### Number of Rows and Columns

In [533]:
df_sold.shape

(591115, 87)

The the sold transactions file has 591115 rows and 87 columns.

##### Review column data types

In [534]:
df_sold.info()

<class 'pandas.DataFrame'>
RangeIndex: 591115 entries, 0 to 591114
Data columns (total 87 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   BuyerAgentAOR                 519608 non-null  str    
 1   ListAgentAOR                  523069 non-null  str    
 2   Flooring                      347464 non-null  str    
 3   ViewYN                        531925 non-null  object 
 4   WaterfrontYN                  322 non-null     object 
 5   BasementYN                    9738 non-null    object 
 6   PoolPrivateYN                 516977 non-null  object 
 7   OriginalListPrice             589404 non-null  float64
 8   ListingKey                    591115 non-null  int64  
 9   ListAgentEmail                547714 non-null  str    
 10  CloseDate                     591115 non-null  str    
 11  ClosePrice                    591108 non-null  float64
 12  ListAgentFirstName            587682 non-null  str    


In [535]:
df_sold.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'WaterfrontYN',
       'BasementYN', 'PoolPrivateYN', 'OriginalListPrice', 'ListingKey',
       'ListAgentEmail', 'CloseDate', 'ClosePrice', 'ListAgentFirstName',
       'ListAgentLastName', 'Latitude', 'Longitude', 'UnparsedAddress',
       'PropertyType', 'LivingArea', 'ListPrice', 'DaysOnMarket',
       'ListOfficeName', 'BuyerOfficeName', 'CoListOfficeName',
       'ListAgentFullName', 'CoListAgentFirstName', 'CoListAgentLastName',
       'BuyerAgentMlsId', 'BuyerAgentFirstName', 'BuyerAgentLastName',
       'FireplacesTotal', 'AssociationFeeFrequency', 'AboveGradeFinishedArea',
       'ListingKeyNumeric', 'MLSAreaMajor', 'TaxAnnualAmount',
       'CountyOrParish', 'MlsStatus', 'ElementarySchool', 'AttachedGarageYN',
       'ParkingTotal', 'BuilderName', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'BuyerOfficeAOR', 'YearBuilt', 'StreetNumberNumeric',
       'ListingId', 'BathroomsTotalInteger', 'City', '

In [536]:
df_sold.isna().sum()

BuyerAgentAOR       71507
ListAgentAOR        68046
Flooring           243651
ViewYN              59190
WaterfrontYN       590793
                    ...  
latfilled          495322
lonfilled          495322
Year                    0
year_month              0
rate_30yr_fixed         0
Length: 87, dtype: int64

#### **Missing Value Analysis**

##### Calculate missing counts and percentages per column

In [537]:
missing_sold_counts = df_sold.isnull().sum()
missing_sold_counts

BuyerAgentAOR       71507
ListAgentAOR        68046
Flooring           243651
ViewYN              59190
WaterfrontYN       590793
                    ...  
latfilled          495322
lonfilled          495322
Year                    0
year_month              0
rate_30yr_fixed         0
Length: 87, dtype: int64

In [538]:
missing_sold_percent = (df_sold.isnull().mean()) * 100
missing_sold_percent

BuyerAgentAOR      12.096969
ListAgentAOR       11.511466
Flooring           41.218883
ViewYN             10.013280
WaterfrontYN       99.945527
                     ...    
latfilled          83.794524
lonfilled          83.794524
Year                0.000000
year_month          0.000000
rate_30yr_fixed     0.000000
Length: 87, dtype: float64

In [539]:
missing_summary = pd.DataFrame({
    "missing_sold_counts": missing_sold_counts,
    "missing_sold_percent": missing_sold_percent
})

In [540]:
missing_sold_summary = missing_summary.sort_values(by="missing_sold_percent", ascending=False)
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115                 100.0
AboveGradeFinishedArea                     591115                 100.0
MiddleOrJuniorSchoolDistrict               591115                 100.0
ElementarySchoolDistrict                   591115                 100.0
FireplacesTotal                            591115                 100.0
...                                           ...                   ...
MlsStatus                                       0                   0.0
StateOrProvince                                 0                   0.0
Year                                            0                   0.0
year_month                                      0                   0.0
rate_30yr_fixed                                 0                   0.0

[87 rows x 2 columns]


In [541]:
missing_sold_summary = missing_sold_summary[missing_sold_summary["missing_sold_counts"] > 0]
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115            100.000000
AboveGradeFinishedArea                     591115            100.000000
MiddleOrJuniorSchoolDistrict               591115            100.000000
ElementarySchoolDistrict                   591115            100.000000
FireplacesTotal                            591115            100.000000
...                                           ...                   ...
PostalCode                                    165              0.027913
ListAgentFullName                             161              0.027237
ListingContractDate                            83              0.014041
ListAgentLastName                              61              0.010319
ClosePrice                                      7              0.001184

[74 rows x 2 columns]


In [542]:
missing_sold_summary["missing_sold_percent"] = missing_sold_summary["missing_sold_percent"].round(2)
print(missing_sold_summary)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115                100.00
AboveGradeFinishedArea                     591115                100.00
MiddleOrJuniorSchoolDistrict               591115                100.00
ElementarySchoolDistrict                   591115                100.00
FireplacesTotal                            591115                100.00
...                                           ...                   ...
PostalCode                                    165                  0.03
ListAgentFullName                             161                  0.03
ListingContractDate                            83                  0.01
ListAgentLastName                              61                  0.01
ClosePrice                                      7                  0.00

[74 rows x 2 columns]


In [543]:
missing_above_90 = missing_sold_summary[missing_sold_summary['missing_sold_percent'] > 90]
print(missing_above_90)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115                100.00
AboveGradeFinishedArea                     591115                100.00
MiddleOrJuniorSchoolDistrict               591115                100.00
ElementarySchoolDistrict                   591115                100.00
FireplacesTotal                            591115                100.00
WaterfrontYN                               590793                 99.95
TaxYear                                    590748                 99.94
BusinessType                               589483                 99.72
TaxAnnualAmount                            588608                 99.58
BelowGradeFinishedArea                     588525                 99.56
BasementYN                                 581377                 98.35
BuilderName                                568536                 96.18
LotSizeDimensions                          560102               

In [544]:
missing_above_70 = missing_sold_summary[missing_sold_summary['missing_sold_percent'] > 70]
print(missing_above_70)

                              missing_sold_counts  missing_sold_percent
CoveredSpaces                              591115                100.00
AboveGradeFinishedArea                     591115                100.00
MiddleOrJuniorSchoolDistrict               591115                100.00
ElementarySchoolDistrict                   591115                100.00
FireplacesTotal                            591115                100.00
WaterfrontYN                               590793                 99.95
TaxYear                                    590748                 99.94
BusinessType                               589483                 99.72
TaxAnnualAmount                            588608                 99.58
BelowGradeFinishedArea                     588525                 99.56
BasementYN                                 581377                 98.35
BuilderName                                568536                 96.18
LotSizeDimensions                          560102               

In [545]:
missing_above_70.shape

(28, 2)

In [546]:
missing_above_70

,missing_sold_counts,missing_sold_percent
CoveredSpaces,591115,100.00
AboveGradeFinishedArea,591115,100.00
MiddleOrJuniorSchoolDistrict,591115,100.00
ElementarySchoolDistrict,591115,100.00
FireplacesTotal,591115,100.00
WaterfrontYN,590793,99.95
TaxYear,590748,99.94
BusinessType,589483,99.72
TaxAnnualAmount,588608,99.58
BelowGradeFinishedArea,588525,99.56


Dropping Variables:
- CoveredSpaces                              
- AboveGradeFinishedArea                     
- MiddleOrJuniorSchoolDistrict               
- ElementarySchoolDistrict                   
- FireplacesTotal                            
- WaterfrontYN                               
- TaxYear                                   
- BusinessType                               
- TaxAnnualAmount                            
- BelowGradeFinishedArea                     
- BasementYN                                 
- BuilderName                                
- LotSizeDimensions                          
- CoBuyerAgentFirstName 
- OriginatingSystemName	
- OriginatingSystemSubName	
- ElementarySchool	
- MiddleOrJuniorSchool	
- BuyerAgencyCompensationType
- BuyerAgencyCompensation	
- BuildingAreaTotal	
- HighSchool	
- latfilled	
- lonfilled	
- CoListAgentFirstName	
- CoListAgentLastName	
- CoListOfficeName	
- AssociationFeeFrequency	

To perserve the variables, it would be sufficient if each variable contain less than 70% of missing values in order to use imputation to replace the missingness in the data. 

So, variables with 70% of missing variables will be removed from the dataset because through imputation with variables with a significant number of missingness, it may contribute to bias towards certain values than others. 

In [547]:
df_sold_clean = df_sold.drop(columns=['CoveredSpaces',
'AboveGradeFinishedArea',	
'MiddleOrJuniorSchoolDistrict',
'ElementarySchoolDistrict',
'FireplacesTotal',
'WaterfrontYN',
'TaxYear',
'BusinessType',
'TaxAnnualAmount',	
'BelowGradeFinishedArea',
'BasementYN',	
'BuilderName',	
'LotSizeDimensions',	
'CoBuyerAgentFirstName',	
'OriginatingSystemName',	
'OriginatingSystemSubName',	
'ElementarySchool',	
'MiddleOrJuniorSchool',	
'BuyerAgencyCompensationType',
'BuyerAgencyCompensation',	
'BuildingAreaTotal',	
'HighSchool',	
'latfilled',	
'lonfilled',	
'CoListAgentFirstName',	
'CoListAgentLastName',
'CoListOfficeName',
'AssociationFeeFrequency'	])

In [548]:
df_sold_clean.shape

(591115, 59)

In [549]:
# Calculate percentage for all columns
missing_pct = (df_sold_clean.isna().sum() / len(df_sold_clean)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

SubdivisionName             64.720232
MainLevelBedrooms           46.942981
Flooring                    41.218883
HighSchoolDistrict          33.351209
AssociationFee              30.998029
AttachedGarageYN            27.720833
Stories                     20.017256
Levels                      13.749778
GarageSpaces                12.801401
PoolPrivateYN               12.542060
BuyerAgentAOR               12.096969
NewConstructionYN           11.869433
ListAgentAOR                11.511466
MLSAreaMajor                10.996507
ViewYN                      10.013280
LotSizeAcres                 9.467532
LotSizeSquareFeet            9.228830
LotSizeArea                  9.151519
FireplaceYN                  8.741108
PropertySubType              7.667205
ListAgentEmail               7.342226
LivingArea                   7.058694
BedroomsTotal                6.658434
BathroomsTotalInteger        4.635477
BuyerOfficeAOR               4.311005
YearBuilt                    4.214747
Latitude    

In [550]:
num_cols_with_missing = df_sold_clean.isna().any().sum()

print(f"Number of variables with missing values: {num_cols_with_missing}")

Number of variables with missing values: 46


In [551]:
cols_rows_dropping = missing_pct[(missing_pct < 1) & (missing_pct > 0)]
cols_rows_dropping

OriginalListPrice           0.289453
ClosePrice                  0.001184
ListAgentFirstName          0.580767
ListAgentLastName           0.010319
UnparsedAddress             0.152255
ListPrice                   0.153946
ListAgentFullName           0.027237
BuyerAgentMlsId             0.309584
BuyerAgentFirstName         0.443907
BuyerAgentLastName          0.056842
StreetNumberNumeric         0.247329
City                        0.074436
ContractStatusChangeDate    0.120450
ListingContractDate         0.014041
PostalCode                  0.027913
dtype: float64

In [552]:
cols_rows_dropped = ['OriginalListPrice', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListPrice', 'ListAgentFullName', 'BuyerAgentMlsId',
                'BuyerAgentFirstName', 'BuyerAgentLastName', 'StreetNumberNumeric', 'City', 'ContractStatusChangeDate', 'ListingContractDate', 'PostalCode']

In [553]:
df_sold_clean = df_sold_clean.dropna(subset=['OriginalListPrice', 'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'UnparsedAddress', 'ListPrice', 'ListAgentFullName', 'BuyerAgentMlsId',
                'BuyerAgentFirstName', 'BuyerAgentLastName', 'StreetNumberNumeric', 'City', 'ContractStatusChangeDate', 'ListingContractDate', 'PostalCode'])

In [554]:
df_sold_clean.shape

(579463, 59)

In [555]:
df_sold_clean.shape[0] - df_sold_clean.shape[0]

0

In [556]:
# Calculate percentage for all columns
missing_pct = (df_sold_clean.isna().sum() / len(df_sold_clean)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct[missing_pct > 0].sort_values(ascending=False))

SubdivisionName          64.446910
MainLevelBedrooms        46.157218
Flooring                 41.177435
HighSchoolDistrict       33.361750
AssociationFee           30.451125
AttachedGarageYN         27.695470
Stories                  19.041078
Levels                   12.705039
GarageSpaces             12.635837
BuyerAgentAOR            12.081013
NewConstructionYN        11.651477
PoolPrivateYN            11.572611
ListAgentAOR             11.501166
MLSAreaMajor             10.944098
ViewYN                    9.802351
LotSizeAcres              9.409401
LotSizeSquareFeet         9.169179
LotSizeArea               9.093937
FireplaceYN               8.496832
PropertySubType           7.567179
ListAgentEmail            7.346630
LivingArea                6.842197
BedroomsTotal             6.409037
BathroomsTotalInteger     4.531609
BuyerOfficeAOR            4.184046
YearBuilt                 4.114844
ParkingTotal              2.967230
Latitude                  2.955668
Longitude           

In [557]:
# Create a reference mapping of Names -> Emails
# Drop any rows that have missing emails so the map is 'clean'
email_lookup = df_sold_clean.dropna(subset=['ListAgentEmail']).drop_duplicates(['ListAgentFirstName', 'ListAgentLastName'])

# Set the names as the index to make the 'lookup' possible
email_lookup = email_lookup.set_index(['ListAgentFirstName', 'ListAgentLastName'])['ListAgentEmail']

# Create a function to apply the logic
def fill_missing_emails(row):
    # Only try to fill if the current email is missing
    if pd.isna(row['ListAgentEmail']):
        # Look up the name in our reference map
        return email_lookup.get((row['ListAgentFirstName'], row['ListAgentLastName']), "None")
    return row['ListAgentEmail']

# Apply it to the dataframe
df_sold_clean['ListAgentEmail'] = df_sold_clean.apply(fill_missing_emails, axis=1)

In [558]:
df_sold_clean['ListAgentEmail'].isna().sum()

np.int64(0)

In [559]:
(df_sold_clean['ListAgentEmail'] == 'None').sum()

np.int64(2537)

From the code above, I noticed that there are no missing values for list agent's first and last names, so to determine their emails, I tried to look up any matches by full name to get the list agent's emails for the rows that are missing them. If there are no matches, those emails will be preserved by having a value of "None" (i.e. no email provided).

In [560]:
df_sold_clean.isna().sum()

BuyerAgentAOR                70005
ListAgentAOR                 66645
Flooring                    238608
ViewYN                       56801
PoolPrivateYN                67059
OriginalListPrice                0
ListingKey                       0
ListAgentEmail                   0
CloseDate                        0
ClosePrice                       0
ListAgentFirstName               0
ListAgentLastName                0
Latitude                     17127
Longitude                    16968
UnparsedAddress                  0
PropertyType                     0
LivingArea                   39648
ListPrice                        0
DaysOnMarket                     0
ListOfficeName                   0
BuyerOfficeName               6566
ListAgentFullName                0
BuyerAgentMlsId                  0
BuyerAgentFirstName              0
BuyerAgentLastName               0
ListingKeyNumeric                0
MLSAreaMajor                 63417
CountyOrParish                   0
MlsStatus           

In [561]:
df_sold_clean.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces

#### **Convert date fields to datetime format (CloseDate, PurchaseContractDate, ListingContractDate, ContractStatusChangeDate)**

In [562]:
df_sold_clean['CloseDate'] = pd.to_datetime(df_sold_clean['CloseDate'], errors='coerce')

In [563]:
df_sold_clean['PurchaseContractDate'] = pd.to_datetime(df_sold_clean['PurchaseContractDate'], errors='coerce')

In [564]:
df_sold_clean['ListingContractDate'] = pd.to_datetime(df_sold_clean['ListingContractDate'], errors='coerce')

In [565]:
df_sold_clean['ContractStatusChangeDate'] = pd.to_datetime(df_sold_clean['ContractStatusChangeDate'], errors='coerce')

In [566]:
print(df_sold_clean['PurchaseContractDate'].dtype)

datetime64[us]


#### **Remove unnecessary or redundant columns**

In [567]:
df_sold_clean.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces

In [568]:
df_sold_clean.shape

(579463, 59)

In [569]:
(df_sold_clean['OriginalListPrice'] == df_sold_clean['ListPrice']).value_counts()

True     394066
False    185397
Name: count, dtype: int64

No redundant columns

#### **Outlier Cleaning**

Remove or flag invalid numeric values: ClosePrice <=0, LivingArea <=0, DaysOnMarket < 0, negative Bedrooms or Bathrooms

Removing and inspecting negative values

In [570]:
neg_counts = (df_sold_clean.select_dtypes(include='number') < 0).sum()

In [571]:
neg_counts[neg_counts > 0]

Latitude             5
Longitude       562392
DaysOnMarket        59
ParkingTotal        95
dtype: int64

In [572]:
(neg_counts[neg_counts > 0] / df_sold_clean.shape[0] * 100).sort_values(ascending=False)

Longitude       97.053997
ParkingTotal     0.016394
DaysOnMarket     0.010182
Latitude         0.000863
dtype: float64

ParkingTotal

In [573]:
neg_ParkingTotal = df_sold_clean[df_sold_clean['ParkingTotal'] < 0]
neg_ParkingTotal.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year,year_month,rate_30yr_fixed
6424,Mlslistings,Mlslistings,Wood,True,False,639000.0,1050719913,josie.realtimerealty@gmail.com,2024-01-19,650000.0,...,NaN,False,2.0,North Monterey County Unified,95012,NaN,5147.00,2024,2024-01,6.6425
9489,Mlslistings,Mlslistings,NaN,False,False,899999.0,1047166342,ascenciorealestate@gmail.com,2024-01-03,905000.0,...,NaN,False,0.0,San Jose Unified,95110,NaN,9583.00,2024,2024-01,6.6425
14274,Mlslistings,Mlslistings,"Laminate,Tile,Wood",False,False,1280000.0,1045843310,marytan@compass.com,2024-01-08,1260000.0,...,NaN,False,0.0,Other,95014,448.0,938.00,2024,2024-01,6.6425
14468,Mlslistings,Mlslistings,Laminate,True,NaN,850000.0,1045777028,carmelasoto415@gmail.com,2024-01-23,770000.0,...,NaN,False,1.0,South San Francisco Unified,94080,722.0,301818.53,2024,2024-01,6.6425
15725,Mlslistings,Mlslistings,"Laminate,Tile",False,NaN,779000.0,1044839849,nitortiz@gmail.com,2024-01-08,755000.0,...,NaN,NaN,0.0,Sacramento City Unified,95822,NaN,6098.00,2024,2024-01,6.6425


In [574]:
neg_ParkingTotal.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year,year_month,rate_30yr_fixed
6424,Mlslistings,Mlslistings,Wood,True,False,639000.0,1050719913,josie.realtimerealty@gmail.com,2024-01-19,650000.0,...,NaN,False,2.0,North Monterey County Unified,95012,NaN,5147.00,2024,2024-01,6.6425
9489,Mlslistings,Mlslistings,NaN,False,False,899999.0,1047166342,ascenciorealestate@gmail.com,2024-01-03,905000.0,...,NaN,False,0.0,San Jose Unified,95110,NaN,9583.00,2024,2024-01,6.6425
14274,Mlslistings,Mlslistings,"Laminate,Tile,Wood",False,False,1280000.0,1045843310,marytan@compass.com,2024-01-08,1260000.0,...,NaN,False,0.0,Other,95014,448.0,938.00,2024,2024-01,6.6425
14468,Mlslistings,Mlslistings,Laminate,True,NaN,850000.0,1045777028,carmelasoto415@gmail.com,2024-01-23,770000.0,...,NaN,False,1.0,South San Francisco Unified,94080,722.0,301818.53,2024,2024-01,6.6425
15725,Mlslistings,Mlslistings,"Laminate,Tile",False,NaN,779000.0,1044839849,nitortiz@gmail.com,2024-01-08,755000.0,...,NaN,NaN,0.0,Sacramento City Unified,95822,NaN,6098.00,2024,2024-01,6.6425


In [575]:
df_sold_clean['ParkingTotal'].describe()

count    5.622690e+05
mean     7.697845e+00
std      3.470027e+03
min     -1.430000e+02
25%      2.000000e+00
50%      2.000000e+00
75%      3.000000e+00
max      2.593669e+06
Name: ParkingTotal, dtype: float64

DaysOnMarket

In [576]:
neg_DaysOnMarket = df_sold_clean[df_sold_clean['DaysOnMarket'] < 0]
neg_DaysOnMarket.head()

,BuyerAgentAOR,ListAgentAOR,Flooring,ViewYN,PoolPrivateYN,OriginalListPrice,ListingKey,ListAgentEmail,CloseDate,ClosePrice,...,MainLevelBedrooms,NewConstructionYN,GarageSpaces,HighSchoolDistrict,PostalCode,AssociationFee,LotSizeSquareFeet,Year,year_month,rate_30yr_fixed
31,CoastalMendocino,CoastalMendocino,Tile,True,False,1950000.0,1063453216,kira@mendosir.com,2024-01-22,1950000.0,...,NaN,NaN,2.0,NaN,95437,NaN,74487.6,2024,2024-01,6.6425
826,VenturaCoastal,VenturaCoastal,Vinyl,False,True,2975.0,1055498337,cindyisalways@gmail.com,2024-01-17,2975.0,...,NaN,NaN,2.0,NaN,93041,0.0,3920.0,2024,2024-01,6.6425
1676,PasadenaFoothills,PasadenaFoothills,Laminate,False,False,4000.0,1054112664,arbi.derian@compass.com,2024-01-16,4000.0,...,0.0,False,2.0,NaN,91601,0.0,6098.0,2024,2024-01,6.6425
3838,PasadenaFoothills,PasadenaFoothills,Vinyl,False,False,3950.0,1052391149,amyponce888@gmail.com,2024-01-31,3950.0,...,NaN,False,0.0,NaN,90041,0.0,5661.0,2024,2024-01,6.6425
9426,CaliforniaDesert,CaliforniaDesert,Carpet,True,False,120000.0,1047183932,martyjelmberg@gmail.com,2024-01-12,120000.0,...,NaN,False,NaN,NaN,92241,NaN,800.0,2024,2024-01,6.6425


In [577]:
rows_DaysOnMarket = neg_DaysOnMarket['DaysOnMarket']
rows_DaysOnMarket.head()

31     -36
826    -11
1676    -1
3838    -1
9426    -4
Name: DaysOnMarket, dtype: int64

In [578]:
non_neg = (df_sold_clean['DaysOnMarket'] >= 0) & (df_sold_clean['ParkingTotal'] >= 0)

In [579]:
df_sold_clean = df_sold_clean[non_neg].copy()

In [580]:
df_sold_clean.shape

(562117, 59)

In [581]:
neg_counts = (df_sold_clean.select_dtypes(include='number') < 0).sum()

In [582]:
neg_counts[neg_counts > 0]

Latitude          3
Longitude    545095
dtype: int64

In [583]:
df_listings_clean = df_sold_clean[
    (df_sold_clean['Latitude'] >= 32.5) & (df_sold_clean['Latitude'] <= 42.0) &
    (df_sold_clean['Longitude'] >= -124.5) & (df_sold_clean['Longitude'] <= -114.0)
]

In [584]:
df_sold_clean.shape

(562117, 59)

In [585]:
'''
df_map = df_sold_clean.sample(30000)

fig = px.scatter_mapbox(
    df_map,
    lat="Latitude",
    lon="Longitude",
    color="OriginalListPrice",
    size_max=15,
    zoom=5,
    title="California Property Value Distribution",
    color_continuous_scale=px.colors.sequential.Plasma,
    mapbox_style="open-street-map"
)

fig.show(renderer='browser')
'''

'\ndf_map = df_sold_clean.sample(30000)\n\nfig = px.scatter_mapbox(\n    df_map,\n    lat="Latitude",\n    lon="Longitude",\n    color="OriginalListPrice",\n    size_max=15,\n    zoom=5,\n    title="California Property Value Distribution",\n    color_continuous_scale=px.colors.sequential.Plasma,\n    mapbox_style="open-street-map"\n)\n\nfig.show(renderer=\'browser\')\n'

#### **Further Handling Missing Values Appropriately**

In [586]:
missing_sold_counts = df_sold_clean.isnull().sum()
missing_sold_counts

BuyerAgentAOR                67632
ListAgentAOR                 64273
Flooring                    221366
ViewYN                       53404
PoolPrivateYN                49793
OriginalListPrice                0
ListingKey                       0
ListAgentEmail                   0
CloseDate                        0
ClosePrice                       0
ListAgentFirstName               0
ListAgentLastName                0
Latitude                     17094
Longitude                    16935
UnparsedAddress                  0
PropertyType                     0
LivingArea                   22454
ListPrice                        0
DaysOnMarket                     0
ListOfficeName                   0
BuyerOfficeName               6535
ListAgentFullName                0
BuyerAgentMlsId                  0
BuyerAgentFirstName              0
BuyerAgentLastName               0
ListingKeyNumeric                0
MLSAreaMajor                 61024
CountyOrParish                   0
MlsStatus           

In [587]:
missing_sold_percent = (df_sold_clean.isnull().mean()) * 100
missing_sold_percent

BuyerAgentAOR               12.031659
ListAgentAOR                11.434096
Flooring                    39.380769
ViewYN                       9.500513
PoolPrivateYN                8.858120
OriginalListPrice            0.000000
ListingKey                   0.000000
ListAgentEmail               0.000000
CloseDate                    0.000000
ClosePrice                   0.000000
ListAgentFirstName           0.000000
ListAgentLastName            0.000000
Latitude                     3.041004
Longitude                    3.012718
UnparsedAddress              0.000000
PropertyType                 0.000000
LivingArea                   3.994542
ListPrice                    0.000000
DaysOnMarket                 0.000000
ListOfficeName               0.000000
BuyerOfficeName              1.162569
ListAgentFullName            0.000000
BuyerAgentMlsId              0.000000
BuyerAgentFirstName          0.000000
BuyerAgentLastName           0.000000
ListingKeyNumeric            0.000000
MLSAreaMajor

In [588]:
missing_summary = pd.DataFrame({
    "missing_sold_counts": missing_sold_counts,
    "missing_sold_percent": missing_sold_percent
})

In [589]:
missing_sold_summary = missing_summary.sort_values(by="missing_sold_percent", ascending=False)
print(missing_sold_summary)

                          missing_sold_counts  missing_sold_percent
SubdivisionName                        357879             63.666283
MainLevelBedrooms                      250160             44.503191
Flooring                               221366             39.380769
HighSchoolDistrict                     176108             31.329421
AssociationFee                         172509             30.689163
AttachedGarageYN                       143275             25.488466
Stories                                 93051             16.553671
BuyerAgentAOR                           67632             12.031659
ListAgentAOR                            64273             11.434096
NewConstructionYN                       64122             11.407234
MLSAreaMajor                            61024             10.856103
Levels                                  56336             10.022113
GarageSpaces                            56028              9.967320
LotSizeAcres                            54336   

In [590]:
missing_sold_summary[missing_sold_summary['missing_sold_percent'] > 0]

,missing_sold_counts,missing_sold_percent
SubdivisionName,357879,63.666283
MainLevelBedrooms,250160,44.503191
Flooring,221366,39.380769
HighSchoolDistrict,176108,31.329421
AssociationFee,172509,30.689163
AttachedGarageYN,143275,25.488466
Stories,93051,16.553671
BuyerAgentAOR,67632,12.031659
ListAgentAOR,64273,11.434096
NewConstructionYN,64122,11.407234


- Under 10% missing -> simple statistical imputation method, but for categorical variables, may include a "Missing" category if deemed necessary
- 10-50% missing -> "Decision Zone" - meaning that there are different approaches that can be taken to impute the missing values like a group-based imputation, model-based if justifiable, adding a missing indicator, or doing simple imputation if necessary
- above 50% -> Since at this point, they are probably important features, the missingness in the data could be important and signals themselves. 

Let's start with "Under 10% missing"

In [591]:
under_10 = missing_sold_summary[(missing_sold_summary['missing_sold_percent'] < 10) & (missing_sold_percent != 0)]

C:\Users\mayab\AppData\Local\Temp\ipykernel_27752\637377579.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  under_10 = missing_sold_summary[(missing_sold_summary['missing_sold_percent'] < 10) & (missing_sold_percent != 0)]


In [592]:
under_10

,missing_sold_counts,missing_sold_percent
GarageSpaces,56028,9.967320
LotSizeAcres,54336,9.666315
ViewYN,53404,9.500513
LotSizeSquareFeet,52940,9.417968
LotSizeArea,52507,9.340938
PoolPrivateYN,49793,8.858120
FireplaceYN,32034,5.698814
PropertySubType,29867,5.313307
BuyerOfficeAOR,24184,4.302307
LivingArea,22454,3.994542


In [593]:
df_sold_clean['GarageSpaces'].value_counts().head()

GarageSpaces
2.0    297989
0.0     80761
1.0     62557
3.0     53120
4.0      7316
Name: count, dtype: int64

In [594]:
df_sold_clean['GarageSpaces'].value_counts(normalize=True).head()

GarageSpaces
2.0    0.588808
0.0    0.159579
1.0    0.123609
3.0    0.104962
4.0    0.014456
Name: proportion, dtype: float64

In [595]:
df_sold_clean['GarageSpaces_Missing'] = df_sold_clean['GarageSpaces'].isna().astype(int)

In [596]:
df_sold_clean['GarageSpaces'] = df_sold_clean['GarageSpaces'].fillna(df_sold_clean['GarageSpaces'].median())

In [597]:
df_sold_clean['GarageSpaces'].value_counts().head()

GarageSpaces
2.0    354017
0.0     80761
1.0     62557
3.0     53120
4.0      7316
Name: count, dtype: int64

In [598]:
df_sold_clean['GarageSpaces'].value_counts(normalize=True).head()

GarageSpaces
2.0    0.629792
0.0    0.143673
1.0    0.111288
3.0    0.094500
4.0    0.013015
Name: proportion, dtype: float64

In [599]:
df_sold_clean['LotSizeAcres'].describe()

count    5.077810e+05
mean     8.010685e+02
std      5.197942e+05
min      0.000000e+00
25%      1.180000e-01
50%      1.653000e-01
75%      2.755000e-01
max      3.702600e+08
Name: LotSizeAcres, dtype: float64

In [600]:
df_sold_clean['LotSizeAcres'] = df_sold_clean['LotSizeAcres'].fillna(df_sold_clean['LotSizeAcres'].median())

In [601]:
df_sold_clean['LotSizeAcres'].isna().sum()

np.int64(0)

The mean is greater than the median. The values are skewed to the right. I will be using the median to impute.

In [602]:
df_sold_clean['ViewYN'].value_counts()

ViewYN
True     307189
False    201524
Name: count, dtype: int64

In [603]:
df_sold_clean['ViewYN'].value_counts(normalize=True)

ViewYN
True     0.603855
False    0.396145
Name: proportion, dtype: float64

In [604]:
df_sold_clean['ViewYN'] = df_sold_clean['ViewYN'].fillna(df_sold_clean['ViewYN'].mode()[0])

In [605]:
df_sold_clean['ViewYN'].value_counts(normalize=True)

ViewYN
True     0.641491
False    0.358509
Name: proportion, dtype: float64

In [606]:
df_sold_clean['LotSizeSquareFeet'].describe()

count    5.091770e+05
mean     4.633343e+05
std      2.065512e+07
min      0.000000e+00
25%      5.105000e+03
50%      7.200000e+03
75%      1.198400e+04
max      5.193920e+09
Name: LotSizeSquareFeet, dtype: float64

The mean is less than the median. The values are skewed to the left. I will be using the median to impute the missing values. 

In [607]:
df_sold_clean['LotSizeSquareFeet'] = df_sold_clean['LotSizeSquareFeet'].fillna(df_sold_clean['LotSizeSquareFeet'].median())

In [608]:
df_sold_clean['LotSizeSquareFeet'].isna().sum()

np.int64(0)

In [609]:
df_sold_clean['LotSizeArea'].describe()

count    5.096100e+05
mean     4.413926e+04
std      2.289552e+06
min      0.000000e+00
25%      4.924000e+03
50%      7.000000e+03
75%      1.100000e+04
max      9.187423e+08
Name: LotSizeArea, dtype: float64

The mean is less than the median. The values are skewed to the left. I will be using the median to impute.

In [610]:
df_sold_clean['LotSizeArea'] = df_sold_clean['LotSizeArea'].fillna(df_sold_clean['LotSizeArea'].median())

In [611]:
df_sold_clean['LotSizeArea'].isna().sum()

np.int64(0)

In [612]:
df_sold_clean['PoolPrivateYN'].value_counts()

PoolPrivateYN
False    452156
True      60168
Name: count, dtype: int64

In [613]:
df_sold_clean['PoolPrivateYN'].value_counts(normalize=True)

PoolPrivateYN
False    0.882559
True     0.117441
Name: proportion, dtype: float64

In [614]:
df_sold_clean['PoolPrivateYN'] = df_sold_clean['PoolPrivateYN'].fillna(df_sold_clean['PoolPrivateYN'].mode()[0])

In [615]:
df_sold_clean['PoolPrivateYN'].value_counts(normalize=True)

PoolPrivateYN
False    0.892962
True     0.107038
Name: proportion, dtype: float64

In [616]:
df_sold_clean['FireplaceYN'].value_counts()

FireplaceYN
True     325132
False    204951
Name: count, dtype: int64

In [617]:
df_sold_clean['FireplaceYN'].value_counts(normalize=True)

FireplaceYN
True     0.613361
False    0.386639
Name: proportion, dtype: float64

In [618]:
df_sold_clean['FireplaceYN'] = df_sold_clean['FireplaceYN'].fillna(df_sold_clean['FireplaceYN'].mode()[0])

In [619]:
df_sold_clean['FireplaceYN'].value_counts(normalize=True)

FireplaceYN
True     0.635394
False    0.364606
Name: proportion, dtype: float64

In [620]:
df_sold_clean['PropertySubType'].value_counts().head()

PropertySubType
SingleFamilyResidence    354434
Condominium              101487
Townhouse                 33508
Apartment                 13203
Duplex                    11020
Name: count, dtype: int64

In [621]:
df_sold_clean['PropertySubType'].value_counts(normalize=True).head()

PropertySubType
SingleFamilyResidence    0.665916
Condominium              0.190675
Townhouse                0.062955
Apartment                0.024806
Duplex                   0.020705
Name: proportion, dtype: float64

In [622]:
df_sold_clean['PropertySubType'] = df_sold_clean['PropertySubType'].fillna(df_sold_clean['PropertySubType'].mode()[0])

In [623]:
df_sold_clean['PropertySubType'].value_counts(normalize=True).head()

PropertySubType
SingleFamilyResidence    0.683667
Condominium              0.180544
Townhouse                0.059610
Apartment                0.023488
Duplex                   0.019604
Name: proportion, dtype: float64

In [624]:
df_sold_clean['BuyerOfficeAOR'].value_counts(normalize=True).head()

BuyerOfficeAOR
OrangeCounty    0.097659
PacificWest     0.068953
SanDiego        0.066027
Southland       0.059626
Mlslistings     0.045723
Name: proportion, dtype: float64

In [625]:
df_sold_clean['BuyerOfficeAOR'] = df_sold_clean['BuyerOfficeAOR'].fillna("Missing")

In [626]:
df_sold_clean['BuyerOfficeAOR'].value_counts(normalize=True).head()

BuyerOfficeAOR
OrangeCounty    0.093457
PacificWest     0.065986
SanDiego        0.063186
Southland       0.057061
Mlslistings     0.043756
Name: proportion, dtype: float64

In [627]:
df_sold_clean['LivingArea'].describe()

count    5.396630e+05
mean     3.525621e+03
std      1.237717e+06
min      0.000000e+00
25%      1.182500e+03
50%      1.578000e+03
75%      2.148000e+03
max      9.090909e+08
Name: LivingArea, dtype: float64

The mean is greater than the median, which shows that it is skewing to the right. I will then impute by the median rather than the mean.

In [628]:
df_sold_clean['LivingArea'] = df_sold_clean['LivingArea'].fillna(df_sold_clean['LivingArea'].median())

In [629]:
df_sold_clean['LivingArea'].describe()

count    5.621170e+05
mean     3.447823e+03
std      1.212745e+06
min      0.000000e+00
25%      1.200000e+03
50%      1.578000e+03
75%      2.113000e+03
max      9.090909e+08
Name: LivingArea, dtype: float64

In [630]:
df_sold_clean['LivingArea'].isna().sum()

np.int64(0)

In [631]:
df_sold_clean['BedroomsTotal'].value_counts().head()

BedroomsTotal
3.0    202621
4.0    128110
2.0    127707
5.0     37197
1.0     31993
Name: count, dtype: int64

In [632]:
df_sold_clean['BedroomsTotal'].value_counts(normalize=True).head()

BedroomsTotal
3.0    0.373717
4.0    0.236288
2.0    0.235544
5.0    0.068607
1.0    0.059008
Name: proportion, dtype: float64

In [633]:
df_sold_clean['BedroomsTotal_Missing'] = df_sold_clean['BedroomsTotal'].isna().astype(int)

In [634]:
df_sold_clean['BedroomsTotal'] = df_sold_clean['BedroomsTotal'].fillna(df_sold_clean['BedroomsTotal'].median())

In [635]:
df_sold_clean['BedroomsTotal'].isna().sum()

np.int64(0)

In [636]:
df_sold_clean['Latitude'].describe()

count    545023.000000
mean         34.482919
std           1.571237
min        -117.472493
25%          33.732933
50%          34.027438
75%          34.263618
max         157.000000
Name: Latitude, dtype: float64

The median and the mean are about the same measure, so I will use the mean to impute missing values.

In [637]:
df_sold_clean['Latitude'] = df_sold_clean['Latitude'].fillna(df_sold_clean['Latitude'].mean())

In [638]:
df_sold_clean['Latitude'].isna().sum()

np.int64(0)

In [639]:
df_sold_clean['Longitude'].describe()

count    545182.000000
mean       -118.429693
std           2.922170
min        -177.646696
25%        -118.514497
50%        -118.038832
75%        -117.332870
max         329.000000
Name: Longitude, dtype: float64

The median and the mean are about the same measure, so I will use the mean to impute missing values.

In [640]:
df_sold_clean['Longitude'] = df_sold_clean['Longitude'].fillna(df_sold_clean['Longitude'].mean())

In [641]:
df_sold_clean['Longitude'].isna().sum()

np.int64(0)

In [642]:
df_sold_clean['PurchaseContractDate'].isna().sum()

np.int64(11725)

In [643]:
df_sold_clean['No_Purchase_Date'] = df_sold_clean['PurchaseContractDate'].isna()

In [644]:
df_sold_clean['YearBuilt'].describe()

count    552912.000000
mean       1978.202358
std          27.062660
min        1776.000000
25%        1960.000000
50%        1979.000000
75%        1999.000000
max        2026.000000
Name: YearBuilt, dtype: float64

The mean and the median are about the same

In [645]:
df_sold_clean['YearBuilt'] = df_sold_clean['YearBuilt'].fillna(round(df_sold_clean['YearBuilt'].mean()))

In [646]:
(df_sold_clean['YearBuilt'] % 1 == 0).value_counts()

YearBuilt
True    562117
Name: count, dtype: int64

In [647]:
df_sold_clean['BathroomsTotalInteger'].describe()

count    553052.000000
mean          2.449177
std           1.415291
min           0.000000
25%           2.000000
50%           2.000000
75%           3.000000
max         175.000000
Name: BathroomsTotalInteger, dtype: float64

In [648]:
df_sold_clean['BathroomsTotalInteger'].value_counts(normalize=True).head()

BathroomsTotalInteger
2.0    0.416420
3.0    0.308387
1.0    0.140504
4.0    0.070181
5.0    0.025415
Name: proportion, dtype: float64

In [649]:
df_sold_clean['BathroomsTotalInteger'] = df_sold_clean['BathroomsTotalInteger'].fillna(round(df_sold_clean['BathroomsTotalInteger'].mean()))

In [650]:
df_sold_clean['BathroomsTotalInteger'].value_counts(normalize=True).head()

BathroomsTotalInteger
2.0    0.425831
3.0    0.303414
1.0    0.138238
4.0    0.069050
5.0    0.025005
Name: proportion, dtype: float64

In [651]:
df_sold_clean['BuyerOfficeName'].value_counts(normalize=True).head()

BuyerOfficeName
Compass                   0.061987
Coldwell Banker Realty    0.037456
NONMEMBER MRML            0.019826
Keller Williams Realty    0.014365
None MRML                 0.014176
Name: proportion, dtype: float64

In [652]:
df_sold_clean['BuyerOfficeName'] = df_sold_clean['BuyerOfficeName'].fillna("Missing")

In [653]:
df_sold_clean['BuyerOfficeName'].value_counts(normalize=True).head()

BuyerOfficeName
Compass                   0.061267
Coldwell Banker Realty    0.037021
NONMEMBER MRML            0.019596
Keller Williams Realty    0.014198
None MRML                 0.014011
Name: proportion, dtype: float64

In [654]:
df_sold_clean['BuyerOfficeName'].isna().sum()

np.int64(0)

Onto the 10-50% missing

In [655]:
btwn_10_50 = missing_sold_summary[(missing_sold_summary['missing_sold_percent'] >= 10) & (missing_sold_percent <= 50)]

C:\Users\mayab\AppData\Local\Temp\ipykernel_27752\1507415290.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  btwn_10_50 = missing_sold_summary[(missing_sold_summary['missing_sold_percent'] >= 10) & (missing_sold_percent <= 50)]


In [656]:
btwn_10_50

,missing_sold_counts,missing_sold_percent
MainLevelBedrooms,250160,44.503191
Flooring,221366,39.380769
HighSchoolDistrict,176108,31.329421
AssociationFee,172509,30.689163
AttachedGarageYN,143275,25.488466
Stories,93051,16.553671
BuyerAgentAOR,67632,12.031659
ListAgentAOR,64273,11.434096
NewConstructionYN,64122,11.407234
MLSAreaMajor,61024,10.856103


In [657]:
df_sold_clean['MainLevelBedrooms'].describe()

count    311957.000000
mean          1.932023
std           6.268009
min           0.000000
25%           1.000000
50%           2.000000
75%           3.000000
max        3333.000000
Name: MainLevelBedrooms, dtype: float64

In [658]:
df_sold_clean['MainLevelBedrooms'].value_counts(normalize=True).head()

MainLevelBedrooms
3.0    0.251798
1.0    0.249156
2.0    0.190311
0.0    0.183532
4.0    0.107076
Name: proportion, dtype: float64

In [659]:
df_sold_clean['MainLevelBedrooms_Missing'] = df_sold_clean['MainLevelBedrooms'].isna().astype(int)

In [660]:
df_sold_clean['MainLevelBedrooms'] = df_sold_clean['MainLevelBedrooms'].fillna(df_sold_clean['MainLevelBedrooms'].median())

In [661]:
df_sold_clean['MainLevelBedrooms'].value_counts(normalize=True).head()

MainLevelBedrooms
2.0    0.550649
3.0    0.139740
1.0    0.138274
0.0    0.101854
4.0    0.059424
Name: proportion, dtype: float64

The mean and the median is about the same, so using mean to impute the missing values but also add a missing indicator.

In [662]:
df_sold_clean['MainLevelBedrooms'].isna().sum()

np.int64(0)

In [663]:
df_sold_clean['Flooring'].value_counts(normalize=True).head()

Flooring
Wood           0.112079
Laminate       0.109206
Carpet,Tile    0.099037
Tile,Wood      0.076531
Vinyl          0.054926
Name: proportion, dtype: float64

In [664]:
df_sold_clean['Flooring'] = df_sold_clean['Flooring'].fillna("Missing")

In [665]:
df_sold_clean['Flooring'].value_counts().head()

Flooring
Missing        221366
Wood            38191
Laminate        37212
Carpet,Tile     33747
Tile,Wood       26078
Name: count, dtype: int64

In [666]:
df_sold_clean['Flooring'].isna().sum()

np.int64(0)

In [667]:
df_sold_clean['HighSchoolDistrict'].value_counts().head()

HighSchoolDistrict
Los Angeles Unified          37144
Other                        31992
Capistrano Unified           14431
Irvine Unified                9754
Saddleback Valley Unified     8490
Name: count, dtype: int64

In [668]:
df_sold_clean['HighSchoolDistrict'] = df_sold_clean['HighSchoolDistrict'].fillna("Missing")

In [669]:
df_sold_clean['HighSchoolDistrict'].value_counts().head()

HighSchoolDistrict
Missing                176108
Los Angeles Unified     37144
Other                   31992
Capistrano Unified      14431
Irvine Unified           9754
Name: count, dtype: int64

In [670]:
df_sold_clean['AssociationFee'].describe()

count    389608.000000
mean        189.305783
std        1669.093040
min           0.000000
25%           0.000000
50%           0.000000
75%         300.000000
max      750000.000000
Name: AssociationFee, dtype: float64

In [671]:
# Regression Predictions
def impute_lgbm_numeric(df, target):
    df = df.copy()

    # mask missing values
    missing_mask = df[target].isna()

    if missing_mask.sum() == 0:
        return df

    # split data
    train = df[~missing_mask]
    test = df[missing_mask]

    y_train = train[target]
    X_train = train.drop(columns=[target]).select_dtypes(include='number')
    X_test = test.drop(columns=[target]).select_dtypes(include='number')

    # encode categorical columns safely
    for col in X_train.select_dtypes(include=['object', 'category']).columns:
        X_train[col] = X_train[col].astype('category').cat.codes
        X_test[col] = X_test[col].astype('category').cat.codes

    # model
    model = LGBMRegressor(
        n_estimators=200,
        learning_rate=0.05,
        random_state=42,
        verbosity=-1
    )

    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    # constraint: association_fee cannot be negative
    preds = np.clip(preds, a_min=0, a_max=None)

    df.loc[missing_mask, target] = preds

    return df

In [672]:
df_sold_clean = impute_lgbm_numeric(df_sold_clean, "AssociationFee")

In [673]:
df_sold_clean['AssociationFee'].describe()

count    562117.000000
mean        201.992750
std        1419.919517
min           0.000000
25%           0.000000
50%          85.000000
75%         276.509215
max      750000.000000
Name: AssociationFee, dtype: float64

In [674]:
df_sold_clean['AssociationFee'].isna().sum()

np.int64(0)

In [675]:
(df_sold_clean['AssociationFee'] < 0).value_counts()

AssociationFee
False    562117
Name: count, dtype: int64

In [676]:
df_sold_clean['AttachedGarageYN'].value_counts()

AttachedGarageYN
True     326539
False     92303
Name: count, dtype: int64

In [677]:
df_sold_clean['AttachedGarageYN'] = df_sold_clean['AttachedGarageYN'].fillna("Missing")

In [678]:
df_sold_clean['AttachedGarageYN'].value_counts()

AttachedGarageYN
True       326539
Missing    143275
False       92303
Name: count, dtype: int64

In [679]:
df_sold_clean['Stories'].describe()

count    469066.000000
mean          1.364936
std           0.481413
min           1.000000
25%           1.000000
50%           1.000000
75%           2.000000
max           2.000000
Name: Stories, dtype: float64

In [680]:
df_sold_clean['Stories'].value_counts()

Stories
1.0    297887
2.0    171179
Name: count, dtype: int64

In [681]:
df_sold_clean['Stories'].value_counts(normalize=True)

Stories
1.0    0.635064
2.0    0.364936
Name: proportion, dtype: float64

In [682]:
df_sold_clean['Stories_Missing'] = df_sold_clean['Stories'].isna().astype(int)

In [683]:
df_sold_clean['Stories'] = df_sold_clean['Stories'].fillna(df_sold_clean['Stories'].median())

In [684]:
df_sold_clean['Stories'].value_counts()

Stories
1.0    390938
2.0    171179
Name: count, dtype: int64

In [685]:
df_sold_clean['Stories'].value_counts(normalize=True)

Stories
1.0    0.695474
2.0    0.304526
Name: proportion, dtype: float64

In [686]:
df_sold_clean['BuyerAgentAOR'].value_counts().head()

BuyerAgentAOR
OrangeCounty             49818
BeverlyHillsGreaterLa    33920
PacificWest              33559
SanDiego                 31522
Southland                31421
Name: count, dtype: int64

In [687]:
df_sold_clean['BuyerAgentAOR'] = df_sold_clean['BuyerAgentAOR'].fillna("Missing")

In [688]:
df_sold_clean['BuyerAgentAOR'].value_counts().head()

BuyerAgentAOR
Missing                  67632
OrangeCounty             49818
BeverlyHillsGreaterLa    33920
PacificWest              33559
SanDiego                 31522
Name: count, dtype: int64

In [689]:
df_sold_clean['ListAgentAOR'].value_counts().head()

ListAgentAOR
OrangeCounty             50215
BeverlyHillsGreaterLa    33985
PacificWest              33787
Southland                32055
SanDiego                 31586
Name: count, dtype: int64

In [690]:
df_sold_clean['ListAgentAOR'] = df_sold_clean['ListAgentAOR'].fillna("Missing")

In [691]:
df_sold_clean['ListAgentAOR'].value_counts().head()

ListAgentAOR
Missing                  64273
OrangeCounty             50215
BeverlyHillsGreaterLa    33985
PacificWest              33787
Southland                32055
Name: count, dtype: int64

In [692]:
df_sold_clean['NewConstructionYN'].value_counts()

NewConstructionYN
False    477369
True      20626
Name: count, dtype: int64

In [693]:
df_sold_clean['NewConstructionYN'].value_counts(normalize=True)

NewConstructionYN
False    0.958582
True     0.041418
Name: proportion, dtype: float64

In [694]:
df_sold_clean['NewConstructionYN'] = df_sold_clean['NewConstructionYN'].fillna(df_sold_clean['NewConstructionYN'].mode()[0])

In [695]:
df_sold_clean['NewConstructionYN'].value_counts(normalize=True)

NewConstructionYN
False    0.963307
True     0.036693
Name: proportion, dtype: float64

In [696]:
df_sold_clean['MLSAreaMajor'].value_counts().head()

MLSAreaMajor
699 - Not Defined                     47153
SRCAR - Southwest Riverside County    23991
252 - Riverside                        6551
248 - Corona                           4280
686 - Ontario                          3757
Name: count, dtype: int64

In [697]:
df_sold_clean['MLSAreaMajor'].value_counts(normalize=True).head()

MLSAreaMajor
699 - Not Defined                     0.094100
SRCAR - Southwest Riverside County    0.047877
252 - Riverside                       0.013073
248 - Corona                          0.008541
686 - Ontario                         0.007498
Name: proportion, dtype: float64

In [698]:
df_sold_clean['MLSAreaMajor'].describe()

count                501093
unique                 1112
top       699 - Not Defined
freq                  47153
Name: MLSAreaMajor, dtype: object

In [699]:
df_sold_clean['MLSAreaMajor'] = df_sold_clean['MLSAreaMajor'].fillna("Missing")

In [700]:
df_sold_clean['MLSAreaMajor'].value_counts(normalize=True).head()

MLSAreaMajor
Missing                               0.108561
699 - Not Defined                     0.083885
SRCAR - Southwest Riverside County    0.042680
252 - Riverside                       0.011654
248 - Corona                          0.007614
Name: proportion, dtype: float64

In [701]:
df_sold_clean['Levels'].value_counts().head()

Levels
One            295037
Two            169935
ThreeOrMore     24621
MultiSplit      11554
One,Two          1602
Name: count, dtype: int64

In [702]:
df_sold_clean['Levels'].value_counts(normalize=True).head()

Levels
One            0.583330
Two            0.335985
ThreeOrMore    0.048679
MultiSplit     0.022844
One,Two        0.003167
Name: proportion, dtype: float64

In [703]:
df_sold_clean['Levels'] = df_sold_clean['Levels'].fillna("Missing")

In [704]:
df_sold_clean['Levels'].value_counts(normalize=True).head()

Levels
One            0.524868
Two            0.302313
Missing        0.100221
ThreeOrMore    0.043800
MultiSplit     0.020554
Name: proportion, dtype: float64

In [705]:
df_sold_clean['Levels'].isna().sum()

np.int64(0)

Over 50% missing values

In [706]:
over_50 = missing_sold_summary[missing_sold_summary['missing_sold_percent'] >= 50]

In [707]:
over_50

,missing_sold_counts,missing_sold_percent
SubdivisionName,357879,63.666283


In [708]:
df_sold_clean['SubdivisionName'] = df_sold_clean['SubdivisionName'].fillna("Missing")

In [709]:
df_sold_clean['SubdivisionName'].value_counts(normalize=True).head()

SubdivisionName
Missing               0.636663
Other                 0.010114
Not Applicable-1      0.006292
Not Applicable        0.005447
Leisure World (LW)    0.004672
Name: proportion, dtype: float64

In [717]:
df_sold_clean.isna().sum().sort_values(ascending=False)

PurchaseContractDate         11725
BuyerAgentAOR                    0
Flooring                         0
ListAgentAOR                     0
PoolPrivateYN                    0
                             ...  
GarageSpaces_Missing             0
BedroomsTotal_Missing            0
No_Purchase_Date                 0
MainLevelBedrooms_Missing        0
Stories_Missing                  0
Length: 64, dtype: int64

In [711]:
df_sold_clean.isna().sum().sum()

np.int64(11725)

In [718]:
df_sold_clean.isna().sum().sum() / df_sold_clean.notna().sum().sum()

np.float64(0.0003260226133733558)

In [721]:
df_sold_clean.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces

In [720]:
#df_sold_clean.to_csv("../data/Sold/sold_cleaned.csv", index=False)

In [725]:
df_listings_clean = pd.read_csv("../data/Listing/listings_cleaned.csv")

In [727]:
df_sold_clean = pd.read_csv("../data/Sold/sold_cleaned.csv")

In [726]:
df_listings_clean.columns

Index(['OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'ListAgentFullName',
       'ListingKeyNumeric', 'MLSAreaMajor', 'CountyOrParish', 'MlsStatus',
       'AttachedGarageYN', 'ParkingTotal', 'PropertySubType', 'LotSizeAcres',
       'SubdivisionName', 'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces', 'HighSchoolDistrict', 'PostalCode', 'AssociationFee',
       'LotSizeSquareFeet', 'Year', 'year_month', 'rate_30yr_fixed',
       'BedroomsTotal_Missing', 'YearBuilt_Missing',
       'Bathroo

In [728]:
df_sold_clean.columns

Index(['BuyerAgentAOR', 'ListAgentAOR', 'Flooring', 'ViewYN', 'PoolPrivateYN',
       'OriginalListPrice', 'ListingKey', 'ListAgentEmail', 'CloseDate',
       'ClosePrice', 'ListAgentFirstName', 'ListAgentLastName', 'Latitude',
       'Longitude', 'UnparsedAddress', 'PropertyType', 'LivingArea',
       'ListPrice', 'DaysOnMarket', 'ListOfficeName', 'BuyerOfficeName',
       'ListAgentFullName', 'BuyerAgentMlsId', 'BuyerAgentFirstName',
       'BuyerAgentLastName', 'ListingKeyNumeric', 'MLSAreaMajor',
       'CountyOrParish', 'MlsStatus', 'AttachedGarageYN', 'ParkingTotal',
       'PropertySubType', 'LotSizeAcres', 'SubdivisionName', 'BuyerOfficeAOR',
       'YearBuilt', 'StreetNumberNumeric', 'ListingId',
       'BathroomsTotalInteger', 'City', 'BedroomsTotal',
       'ContractStatusChangeDate', 'PurchaseContractDate',
       'ListingContractDate', 'StateOrProvince', 'FireplaceYN', 'Stories',
       'Levels', 'LotSizeArea', 'MainLevelBedrooms', 'NewConstructionYN',
       'GarageSpaces